# Dubai Real Estate 2025 - Developer Extraction
This notebook documents the creation of a new developer-level feature for Dubai’s 2025 real estate transactions, consolidating information from the heterogeneous text fields Building and Project wherever possible, providing an interpretable and economically meaningful variable for subsequent statistical and predictive analyses.  
Master Project was excluded from developer identification because it frequently represents a master development or geographical location rather than the property developer. Initial validation showed that names matching developer entities in this field could therefore produce false developer assignments.

## Main Steps

**- Text Preprocessing:** All relevant text fields were lowercased, concatenated and normalized to ensure consistent tokenization.  

**- Exact Matching:** Known developer names were identified within the available project, building and master-project fields using exact text matching.

**- Pattern Matching:** Additional developers were extracted from explicit textual patterns (e.g., “by …”, “developed by …”).

**- Manual Mapping:** High-frequency unmatched project combinations were manually mapped using external sources.

**- Entity Consolidation:** Similar developer names were inspected and consolidated where they represented the same company.  

**- Similarity-Based Propagation:** Developer assignments were propagated to highly similar project and building names, with manually defined exclusions for ambiguous matches.

**- Final Integration:** Clean developer names were merged back into the master dataset, replacing null values while preserving the original project identifiers.

## Outcome

**Final developer coverage:** ≈ 69 % of all transactions (up from ~36 % in early automated extraction).  
≈ 68% of coverage for Units  
≈ 75% of coverage for Villas

**Model validation:** inclusion of the developer variable increased adjusted R² from 0.50 → 0.63, and feature importance in a random forest model showed ≈ 20 %. This was performed in a separated notebook for personal research porpouses and is therefore not included.

These results confirm that developer identity is a substantive market driver, capturing brand-, quality- and location-related effects not explained by property type, size or area alone.  
This final version provides a reliable, interpretable and quantitatively validated Developer feature suitable for both explanatory analysis and predictive modeling.

# Libraries

In [1]:
import pandas as pd # Data Manipulation
import numpy as np # Data Manipulation

import re
from rapidfuzz import fuzz
from itertools import combinations

from rapidfuzz import process, fuzz

# Data

In [4]:
raw = pd.read_csv('re_2025_analysis.csv')
df = raw.copy()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220918 entries, 0 to 220917
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_type  220918 non-null  object 
 1   date              220918 non-null  object 
 2   property_type     220918 non-null  object 
 3   registration      220918 non-null  object 
 4   area              220918 non-null  object 
 5   building          194538 non-null  object 
 6   project           197958 non-null  object 
 7   master_project    194085 non-null  object 
 8   landmark          220918 non-null  object 
 9   metro             220918 non-null  object 
 10  mall              220918 non-null  object 
 11  rooms             220918 non-null  object 
 12  parking           220918 non-null  int64  
 13  size              220918 non-null  float64
 14  price             220918 non-null  float64
 15  metre_price       220918 non-null  float64
 16  no_sellers        22

# 1. Data Preparation
The three string features that are used for developer's extraction are preprocessed to be ready for their use.

## 1.1. Creating Subset
Create a new DataFrame with the interesting variables that will be utilised during the 'developer' extraction process.

In [10]:
# Subset and copy (avoid modifying original)
df_dev = df[['building', 'project', 'master_project']].copy()

## 1.2. Normalize Null Values
Most missing values in the dataset have already been dealt with. The missing ones, left on purpose for this section, which are contained in the columns 'building', 'project' and 'master_project' are converted to empty strings to be able to perform the developer extraction techniques.

In [13]:
df_dev = df_dev.fillna('').astype(str)

## 1.3. Normalize Text
Apply basic text normalizing techniques to facilitate the text handling.

In [16]:
# --- Normalize all text ---
def normalize_text(s):
    """
    Clean and standardize strings for text matching.
    - Convert to lowercase
    - Remove excessive whitespace
    - Strip leading/trailing spaces
    """
    s = str(s).lower().strip()
    s = re.sub(r'\s+', ' ', s)       # collapse multiple spaces
    return s

In [18]:
for col in ['building', 'project', 'master_project']:
    df_dev[col] = df_dev[col].apply(normalize_text)

## 1.4. Concatanating Columns

In [16]:
# --- Combine all 3 columns into a single searchable text field ---
# Starting with building
df_dev['proj_concat'] = (
    df_dev['building'] + ' | ' +
    df_dev['project'] + ' | ' +
    df_dev['master_project']
).str.strip()

## 1.5. Deduplication

In [18]:
# --- Deduplication for faster exploration ---
projects_unique = df_dev['proj_concat'].drop_duplicates().reset_index(drop=True)

print(f"Unique project strings: {len(projects_unique)}")
projects_unique.tail(5)

Unique project strings: 4403


4398    ivy at park five | ivy at park five | internat...
4399             | park villa's | jumeirah village circle
4400                 plaza boutique - 4 |  | business bay
4401    urbana iii stacked house block-32 | urbana iii...
4402    park lane ? townhouses | park lane | dubai hil...
Name: proj_concat, dtype: object

In [22]:
projects_unique = df_dev['project'].drop_duplicates().reset_index(drop=True)
buildings_unique = df_dev['building'].drop_duplicates().reset_index(drop=True)
masters_unique = df_dev['master_project'].drop_duplicates().reset_index(drop=True)

# 2. Exact Developer Matching
Create a list of known developer companies operating in Dubai. This list was obtained from online reliable sources.

Developer names are extracted independently from the building, project and master-project fields. Multiple developer candidates were identified for 2,902 transactions. Manual inspection showed that all conflicts involved a developer identified in the building/project fields and a location or master-development name identified in the master-project field. Based on this validation, building-level matches were given priority, followed by project-level and master-project matches.

A coverage of **29.1%** was achieved during this process.

## 2.1. Developer Dictionary and Matching Pattern

In [20]:
# Create a list of Dubai known developers 
developers = [
    "emaar", "damac", "nakheel", "sobha", "meraas", "azizi",
    "aldar", "dubai properties", "union properties", "nshama",
    "deyaar", "majid al futtaim", "mag", "ellington", "select group",
    "binghatti", "danube", "omniyat", "tiger properties", "wasl",
    "meydan", "al habtoor", "arenco", "arady", "rak properties",
    "reportage properties", "dubai holding", "al ghurair",
    "tecom group", "limitless", "seven tides", "dubai south",
    "time properties", "reef real estate", "fam properties",

    "imkan", "imtilak", "arista", "arada", "liwan", "khalifa bin dasmal",
    "skyline builders", "gemini property developers", "azco real estate",
    "lazourde", "oman properties", "samana", "gulf general investments",
    "khamas group", "empire development", "srk real estate", 
    "creative cluster authority", "palma holding", "dar al arkan",
    "daark real estate", "premier developers", "symphony developers",
    "bloom properties", "eagle hills", "liv developers", "tabeer",
    "ramhan island development", "trident", "zaya developers",
    "merlin developers", "eagle properties", "al barari", 'irth',
    "tebyan", "cayan group", "omnia developments", "artar",
    "tanmiyat", "mirage", "decent real estate", "serenia developers",

    "avenew development", "ohana development", "pasha one",
    "valores property development", "richmind development",
    "confident group", "vision developments", "metac",
    "patriot developers", "aqua properties", "centurion",
    "sycamore developments", "ghrei development", "dar global",
    "avenue property", "reva developers", "q development",
    "arabtec", "universal properties", "realty force",
    "signature developers", "the first group", "oriental real estate",
    "driven properties", "skyline developers", "jumeirah golf estates"
]

In [22]:
text_columns = ['building', 'project']

In [24]:
# Create Regex pattern to search for developers
pattern = re.compile(
    r'\b(' + '|'.join(map(re.escape, developers)) + r')\b'
)

## 2.2. Developer Extraction

In [27]:
# Extracting developers from the three columns
def extract_all_developers(text):
    """
    Return all unique developer names found in a text string.
    """
    matches = [m.group(1) for m in pattern.finditer(text)]
    
    # Preserve order while removing duplicates
    return list(dict.fromkeys(matches))

for col in text_columns:
    df_dev[f'{col}_matches'] = df_dev[col].apply(extract_all_developers)

In [29]:
# Combining the results into a new column
def combine_matches(row):
    matches = (
        row['building_matches']
        + row['project_matches']
    )
    
    return list(dict.fromkeys(matches))

df_dev['developer_candidates'] = df_dev.apply(
    combine_matches,
    axis=1
)

## 2.3. Conflict Validation and Resolution
2,902 transactions contained two developer-list matches. Inspection showed that all conflicts involved a developer name identified in the building/project fields and a second name from the master-project field. The latter corresponded to master developments or locations rather than the property developer. No transactions contained three developer candidates.

Based on this validation, the following priority was adopted for exact developer extraction:

Building → Project → Master Project.

In [32]:
# Counting developers conflicts
df_dev['n_developer_candidates'] = (
    df_dev['developer_candidates'].str.len()
)

df_dev['n_developer_candidates'].value_counts().sort_index()

n_developer_candidates
0    163326
1     57137
2       455
Name: count, dtype: int64

In [34]:
# Checking the conflicts
conflicts = df_dev[
    df_dev['n_developer_candidates'] > 1
].copy()

conflicts['developer_candidates'].value_counts()

developer_candidates
[azizi, mirage]        424
[binghatti, mirage]     31
Name: count, dtype: int64

In [36]:
# Assigning the developers found
def resolve_developer(row):
    """
    Select the developer using the validated priority:
    building → project → master_project.
    """
    
    for col in [
        'project_matches'
    ]:
        if row[col]:
            return row[col][0]
    
    return np.nan

df_dev['developer_exact'] = df_dev.apply(
    resolve_developer,
    axis=1
)

print("Exact-match coverage:",
      f"{df_dev['developer_exact'].notna().mean()*100:.1f}% of records")

Exact-match coverage: 25.8% of records


In [38]:
# Dropping temporary columns
df_dev = df_dev[['building', 'project', 'master_project', 'developer_exact']]

# 3. Pattern-Based Extraction

## New

In [42]:
pattern_dev = re.compile(
    r'\bby\s+(.+?)(?:\s*$|\s*\|)',
    flags=re.IGNORECASE
)

In [44]:
def extract_all_pattern_developers(text):
    matches = [
        m.group(1).strip()
        for m in pattern_dev.finditer(text)
    ]
    
    return list(dict.fromkeys(matches))

In [46]:
for col in text_columns:
    df_dev[f'{col}_matches'] = (
        df_dev[col].apply(extract_all_pattern_developers)
    )

In [48]:
df_dev['developer_pattern_candidates'] = df_dev.apply(
    combine_matches,
    axis=1
)

In [50]:
new_pattern_matches = (
    df_dev['developer_exact'].isna()
    & df_dev['developer_pattern_candidates'].str.len().gt(0)
)

print(
    f"New records identified by pattern matching: "
    f"{new_pattern_matches.sum():,}"
)

print(
    f"New coverage from pattern matching: "
    f"{new_pattern_matches.mean() * 100:.1f}%"
)

New records identified by pattern matching: 16,317
New coverage from pattern matching: 7.4%


In [52]:
df_dev[
    df_dev['developer_pattern_candidates'].str.len() > 1
]['developer_pattern_candidates'].value_counts()

developer_pattern_candidates
[damac tower-b, damac]                                    379
[damac tower-a, damac]                                    319
[damac tower-c, damac]                                    301
[vida tower 1, vida]                                      151
[danube - tower a, danube]                                141
[danube - 1, danube]                                      134
[elie saab tower 1, elie saab]                            134
[danube - tower b, danube]                                127
[beyond tower-2, beyond]                                  127
[beyond - tower a, beyond]                                123
[danube - 2, danube]                                      115
[vida tower 2, vida]                                      111
[citi developer, citi developers]                         110
[paramount (d), paramount]                                 82
[damac (b), damac]                                         79
[damac (a), damac]                       

In [54]:
def resolve_pattern_developer(row):
    for col in [
        'building_matches',
        'project_matches'
    ]:
        if row[col]:
            return row[col][0]
    
    return np.nan

In [56]:
df_dev['developer_pattern'] = df_dev.apply(
    resolve_pattern_developer,
    axis=1
)

In [57]:
pattern_conflicts = df_dev[
    df_dev['developer_pattern_candidates'].str.len() > 1
].copy()

pattern_conflicts['exact_in_candidates'] = pattern_conflicts.apply(
    lambda row: (
        row['developer_exact'] in row['developer_pattern_candidates']
        if pd.notna(row['developer_exact'])
        else False
    ),
    axis=1
)

pattern_conflicts['exact_in_candidates'].value_counts()

exact_in_candidates
True     1732
False    1481
Name: count, dtype: int64

In [58]:
new_pattern = df_dev[
    df_dev['developer_exact'].isna()
    & df_dev['developer_pattern_candidates'].str.len().gt(0)
].copy()

In [62]:
new_pattern.shape

(16317, 8)

In [64]:
new_pattern.head()

,building,project,master_project,developer_exact,building_matches,project_matches,developer_pattern_candidates,developer_pattern
150,antalya by karma,antalya by karma,dubai sports city,NaN,[karma],[karma],[karma],karma
211,elevate by prescott,elevate by prescott,arjan,NaN,[prescott],[prescott],[prescott],prescott
212,elevate by prescott,elevate by prescott,arjan,NaN,[prescott],[prescott],[prescott],prescott
214,curve by sentro,,arjan,NaN,[sentro],[],[sentro],sentro
228,riviera lodge,lodge residences by riviera azure,jumeirah village circle,NaN,[],[riviera azure],[riviera azure],riviera azure


In [66]:
pattern_conflicts[
    ~pattern_conflicts['exact_in_candidates']
][
    [
        'building',
        'project',
        'master_project',
        'developer_exact',
        'developer_pattern_candidates'
    ]
].head(50)

,building,project,master_project,developer_exact,developer_pattern_candidates
356,aveline residencies by citi developer,aveline residences by citi developers,jumeirah village circle,NaN,"[citi developer, citi developers]"
357,aveline residencies by citi developer,aveline residences by citi developers,jumeirah village circle,NaN,"[citi developer, citi developers]"
358,aveline residencies by citi developer,aveline residences by citi developers,jumeirah village circle,NaN,"[citi developer, citi developers]"
785,damac towers by paramount (b),damac towers by paramount,business bay,damac,"[paramount (b), paramount]"
786,damac towers by paramount (a),damac towers by paramount,business bay,damac,"[paramount (a), paramount]"
787,damac towers by paramount (a),damac towers by paramount,business bay,damac,"[paramount (a), paramount]"
788,damac towers by paramount (d),damac towers by paramount,business bay,damac,"[paramount (d), paramount]"
923,bellagio by sunrise - building 2,bellagio by sunrise,wasl gate,NaN,"[sunrise - building 2, sunrise]"
924,bellagio by sunrise - building 1,bellagio by sunrise,wasl gate,NaN,"[sunrise - building 1, sunrise]"
925,bellagio by sunrise - building 1,bellagio by sunrise,wasl gate,NaN,"[sunrise - building 1, sunrise]"


In [68]:
new_pattern_single = df_dev[
    df_dev['developer_exact'].isna()
    & df_dev['developer_pattern_candidates'].str.len().eq(1)
].copy()

new_pattern_single[
    [
        'building',
        'project',
        'master_project',
        'developer_pattern_candidates'
    ]
].head(50)

,building,project,master_project,developer_pattern_candidates
150,antalya by karma,antalya by karma,dubai sports city,[karma]
211,elevate by prescott,elevate by prescott,arjan,[prescott]
212,elevate by prescott,elevate by prescott,arjan,[prescott]
214,curve by sentro,,arjan,[sentro]
228,riviera lodge,lodge residences by riviera azure,jumeirah village circle,[riviera azure]
229,one park square by iman,one park square by iman,jumeirah village circle,[iman]
230,one park square by iman,one park square by iman,jumeirah village circle,[iman]
231,one park square by iman,one park square by iman,jumeirah village circle,[iman]
232,one park square by iman,one park square by iman,jumeirah village circle,[iman]
233,one park square by iman,one park square by iman,jumeirah village circle,[iman]


In [70]:
pattern_new = df_dev[
    df_dev['developer_exact'].isna()
    & df_dev['developer_pattern_candidates'].str.len().gt(0)
].copy()

In [72]:
from collections import Counter

candidate_counts = Counter(
    candidate
    for candidates in pattern_new['developer_pattern_candidates']
    for candidate in candidates
)

candidate_frequency = (
    pd.DataFrame(
        candidate_counts.items(),
        columns=['candidate_developer', 'transactions']
    )
    .sort_values('transactions', ascending=False)
    .reset_index(drop=True)
)

candidate_frequency.head(50)

,candidate_developer,transactions
0,imtiaz,2935
1,peace homes,1249
2,iman,971
3,prestige one,872
4,beyond,601
5,prescott,598
6,vision,538
7,kasco,430
8,haven 1,364
9,igo,336


## 3.2. Old

In [23]:
# --- Combine all 3 columns into a single searchable text field ---
# Starting with master_project
df_dev['proj_concat'] = (
    df_dev['master_project'] + ' | ' +
    df_dev['project'] + ' | ' +
    df_dev['building']
).str.strip()

In [24]:
pattern_by_general = re.compile(r'\bby\s+([a-z0-9&\s\-\.]{2,60})', flags=re.IGNORECASE)

def extract_dev_pattern_general(text):
    m = pattern_by_general.search(text)
    if m:
        return m.group(1).strip()
    return np.nan

# Apply on the deduplicated list first (much faster)
projects_unique = df_dev['proj_concat'].drop_duplicates().reset_index(drop=True)
proj_df = pd.DataFrame({'proj_concat': projects_unique})

proj_df['dev_pattern2'] = proj_df['proj_concat'].apply(extract_dev_pattern_general)
df_dev = df_dev.merge(proj_df[['proj_concat','dev_pattern2']], on='proj_concat', how='left')

# Combine: prefer exact match, else pattern match
df_dev['dev_raw'] = df_dev['dev_exact'].fillna(df_dev['dev_pattern2'])

print("Coverage after exact and pattern-based extraction:",
      f"{df_dev['dev_raw'].notna().mean()*100:.1f}%")

Pattern-based new coverage: 36.3% of records


In [25]:
df_dev['dev_raw'].value_counts()

dev_raw
binghatti                              19156
damac                                  10141
sobha                                   8189
azizi                                   5863
danube                                  4971
                                       ...  
london gate real estate development        5
nexus                                      4
al marina                                  4
newbury developments                       1
casa vista development                     1
Name: count, Length: 122, dtype: int64

# 4. Manual Mapping
The most frequent unmatched project/building combinations were manually reviewed and mapped using external sources. The resulting curated mapping was then merged into the dataset.

In [27]:
# Identify unmatched project strings (no developer found after fuzzy and pattern matching)
#unmatched = df_dev[df_dev['dev_raw'].isna()]

In [28]:
# Export top unmatched project strings for manual lookup
#unmatched = df_dev[df_dev['dev_raw'].isna()]
#top_unmatched = unmatched['proj_concat'].value_counts().reset_index().rename(columns={'index':'proj_concat','proj_concat':'count'})
#top_unmatched.to_csv('top_unmatched_projects.csv', index=False)
# print("Exported top_unmatched_projects.csv — inspect top rows.")
# top_unmatched.head(50)

In [29]:
manual_map = pd.read_csv('developer_manual_mapping.csv')
df_dev = df_dev.merge(manual_map, on='proj_concat', how='left')
df_dev['developer_clean'] = (
    df_dev['dev_raw']
    .fillna(df_dev['manual_developer'])
)
coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage after manual enrichment: {coverage:.1f}%")

Final developer coverage after manual enrichment: 54.0%


In [30]:
manual_map2 = pd.read_csv('developer_manual_mapping2.csv').drop(columns=['count'], errors='ignore')

df_dev = df_dev.merge(manual_map2, on='proj_concat', how='left')

df_dev['developer_clean'] = (
    df_dev['developer_clean']
    .fillna(df_dev['manual_developer2'])
)

coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage after manual enrichment: {coverage:.2f}%")

Final developer coverage after manual enrichment: 63.10%


# 5. Developer Name Standardization
Potential duplicate developer labels were identified using string similarity and manually reviewed. Labels representing the same developer were consolidated, while genuinely different developers with similar names were retained separately.

In [32]:
df_dev['developer_clean'].nunique()

222

In [33]:
dev_names = sorted(df_dev['developer_clean'].dropna().unique())
pairs_contained = []

for i, a in enumerate(dev_names):
    for b in dev_names[i+1:]:
        # skip if identical
        if a == b: 
            continue
        # check containment (longer string includes shorter one)
        if a in b or b in a:
            pairs_contained.append((a, b, len(a)/len(b) if len(b)>0 else 0))

contain_df = pd.DataFrame(pairs_contained, columns=['dev1','dev2','length_ratio'])
display(contain_df.head(30))
print(f"Found {len(contain_df)} containment pairs.")

,dev1,dev2,length_ratio
0,citi developers,citi developers - tower 1,0.600000
1,citi developers,citi developers - tower 2,0.600000
2,condor,condor group,0.500000
3,ellington,ellington properties,0.450000
4,ellington,wellington,0.900000
5,fakhruddin,fakhruddin properties,0.476190
6,iman,iman developers,0.266667
7,irth,irth group,0.400000
8,london gate,london gate real estate development,0.314286
9,mag,mag,0.750000


Found 19 containment pairs.


In [34]:
dev_unique = sorted([d.strip().lower() for d in df_dev['developer_clean'].dropna().unique()])

# Create a dataframe of all pairs with similarity score
pairs = []
for a, b in combinations(dev_unique, 2):
    score = fuzz.token_sort_ratio(a, b)
    if score >= 85:  # tune threshold, 85–95 recommended
        pairs.append((a, b, score))

dev_pairs = pd.DataFrame(pairs, columns=['developer_1','developer_2','similarity']).sort_values('similarity', ascending=False)

print(f"Found {len(dev_pairs)} potentially duplicated developer pairs.")
display(dev_pairs.head(30))

Found 29 potentially duplicated developer pairs.


,developer_1,developer_2,similarity
21,mag,mag,100.000000
9,citi developers - tower 1,citi developers - tower 2,96.000000
11,ellington,wellington,94.736842
24,saas properties,saba properties,93.333333
28,tiger properties,time properties,90.322581
19,maaia developers,maas developers,90.322581
25,saba properties,sbk properties,89.655172
16,hre development,hz development,89.655172
13,h&h development,hz development,89.655172
2,asak real estate development,deca real estate development,89.285714


In [35]:
merge_map = {
    'citi developers - tower 1': 'citi developers',
    'citi developers - tower 2': 'citi developers',
    'condor group':'condor',
    'ellington properties':'ellington',
    'fakhruddin properties':'fakhruddin',
    'iman developers': 'iman',
    'irth group': 'irth',
    'london gate real estate development': 'london gate',
    'mag ':'mag',
    'majid al futtaim': 'majid',
    'mashriq elite development': 'mashriq elite',
    'nshama development': 'nshama',
    'prescott real estate development': 'prescott',
    'reportage properties': 'reportage',
    'tabeer developments': 'tabeer',
    'taraf developments': 'taraf',
    'union properties': 'union',
    'zimaya properties': 'zimaya',
    'al dar 1': 'aldar',
    'al dar 2': 'aldar',
    'athlon 1': 'aldar',
    'athlon 2': 'aldar',
    'athlon 3': 'aldar',
    'athlon 4': 'aldar',
    'haven 1': 'haven',
    'haven 2': 'haven' 
}

In [36]:
df_dev['developer_clean'] = df_dev['developer_clean'].replace(merge_map)
df_dev['developer_clean'].nunique()

197

# 6. Similarity-Based Propagation

## 6.1. Projects

In [38]:
# Get known (already matched) and unknown projects
known_projects = (
    df_dev[df_dev['developer_clean'].notna()]
    [['project', 'developer_clean']]
    .drop_duplicates(subset='project')
)
unknown_projects = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['project'].notna() & (df_dev['project'].str.strip() != ""), ['project']]
    .drop_duplicates(subset='project')
)

# For each unknown project, find the most similar known project
matches = []
for proj in unknown_projects['project']:
    match = process.extractOne(
        proj,
        known_projects['project'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=88  # adjust threshold; 
    )
    if match:
        matched_proj, score, _ = match
        matched_dev = known_projects.loc[known_projects['project']==matched_proj, 'developer_clean'].iloc[0]
        matches.append((proj, matched_proj, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_project','matched_project','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched projects have strong similarity (>=88) with known projects.")
display(propagation_df.head())

131 unmatched projects have strong similarity (>=88) with known projects.


,unmatched_project,matched_project,developer_suggested,similarity
0,emirates living - springs 2,emirates living - springs 1,emaar,96.296296
1,emirates living - springs 12,emirates living - springs 1,emaar,98.181818
2,emirates living - springs 9,emirates living - springs 1,emaar,96.296296
3,emirates living - springs 3,emirates living - springs 1,emaar,96.296296
4,emirates living - springs 5,emirates living - springs 1,emaar,96.296296


In [39]:
# your exclusion list
excluded_unmatched = [
    'balqis residence', 'elle residences', 'laya residences',
    'olivia residences', 'liv residence', 'maya townhouses',
    'verdana 2', 'myka residence', 'aria', 'mr.c residences downtown',
    'elevia residences', 'azha downtown residences', 'riva residence',
    'bv residences', 'belmont residences', 'lua residences', 'nb residences'
]

# create propagation mapping excluding those
propagate_map = (
    propagation_df.loc[~propagation_df['unmatched_project'].isin(excluded_unmatched)]
    .set_index('unmatched_project')['developer_suggested']
    .to_dict()
)

# apply the mapping
df_dev['developer_clean'] = df_dev['developer_clean'].fillna(
    df_dev['project'].map(propagate_map)
)

# verify coverage after propagation
coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 66.83%


## 6.2. Buildings

In [41]:
# Get known (already matched) and unknown buildings
known_buildings = (
    df_dev[df_dev['developer_clean'].notna()]
    [['building', 'developer_clean']]
    .drop_duplicates(subset='building')
)
unknown_buildings = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['building'].notna() & (df_dev['building'].str.strip() != ""), ['building']]
    .drop_duplicates(subset='building')
)

# For each unknown building, find the most similar known building
matches = []
for proj in unknown_buildings['building']:
    match = process.extractOne(
        proj,
        known_buildings['building'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90  # adjust threshold; 90 = strong match
    )
    if match:
        matched_proj, score, _ = match
        matched_dev = known_buildings.loc[known_buildings['building']==matched_proj, 'developer_clean'].iloc[0]
        matches.append((proj, matched_proj, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_building','matched_building','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched buildings have strong similarity (>=90) with known buildings.")
display(propagation_df.head())

27 unmatched buildings have strong similarity (>=90) with known buildings.


,unmatched_building,matched_building,developer_suggested,similarity
0,views 1,views 1,dubai south,100.000000
1,elite residences 3,elite residence,tameer holdings,90.909091
2,elite residence 1,elite residence,tameer holdings,93.750000
3,elite residences 2,elite residence,tameer holdings,90.909091
4,south residences,south residence 2,dubai south,90.909091


In [42]:
# your exclusion list
excluded_unmatched = [
    'jade residence', 'riviera residence', 'taya residences',
    'olivia residences', 'liv residence', 'dana tower', 'amalia residences'
]

# create propagation mapping excluding those
propagate_map = (
    propagation_df.loc[~propagation_df['unmatched_building'].isin(excluded_unmatched)]
    .set_index('unmatched_building')['developer_suggested']
    .to_dict()
)

# apply the mapping
df_dev['developer_clean'] = df_dev['developer_clean'].fillna(
    df_dev['building'].map(propagate_map)
)

# verify coverage after propagation
coverage = df_dev['developer_clean'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 67.15%


# 7. Prefix-Based Propagation

## 7.1. Projects

In [44]:
# Function to split project names into prefix and suffix
def split_project_name(name):
    """
    Splits a project name into two parts:
    - prefix: everything before the first non-letter/non-space character
    - suffix: everything after that character
    If no such character exists, suffix is None.
    """
    if pd.isna(name):
        return pd.Series([None, None])
    
    match = re.split(r'[^A-Za-z\s]+', name, maxsplit=1)
    
    if len(match) == 1:
        return pd.Series([match[0].strip(), None])
    else:
        return pd.Series([match[0].strip(), match[1].strip()])

# Apply to your DataFrame
df_dev[['proj_prefix', 'proj_suffix']] = df_dev['project'].apply(split_project_name)

In [45]:
# Get known (already matched) and unknown projects
known_projects = (
    df_dev[df_dev['developer_clean'].notna()]
    [['proj_prefix', 'developer_clean']]
    .drop_duplicates(subset='proj_prefix')
)
unknown_projects = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['proj_prefix'].notna() & (df_dev['proj_prefix'].str.strip() != ""), ['proj_prefix']]
    .drop_duplicates(subset='proj_prefix')
)

# For each unknown project, find the most similar known project
matches = []
for proj in unknown_projects['proj_prefix']:
    match = process.extractOne(
        proj,
        known_projects['proj_prefix'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90  # adjust threshold; 90 = strong match
    )
    if match:
        matched_proj, score, _ = match
        matched_dev = known_projects.loc[known_projects['proj_prefix']==matched_proj, 'developer_clean'].iloc[0]
        matches.append((proj, matched_proj, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_proj_prefix','matched_proj_prefix','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched project-prefixes have strong similarity (>=90) with known project-prefixes.")
display(propagation_df.head())

34 unmatched project-prefixes have strong similarity (>=90) with known project-prefixes.


,unmatched_proj_prefix,matched_proj_prefix,developer_suggested,similarity
0,elite,elite,tameer holdings,100.000000
1,balqis residence,bali residences,serene developments,90.322581
2,avenue residence,avenue residence,nabni,100.000000
3,midtown,midtown,deyaar,100.000000
4,maple,maple,emaar,100.000000


In [46]:
# your exclusion list
excluded_prefixes = [
    'balqis residence', 'elle residences', 'axis residences', 'the', 'amalia residences',
    'laya residences', 'olivia residences', 'verdana', 'liv residence', 'the residence',
    'maya residences', 'maya townhouses', 'serra tower', 'prive residence',
    'f', 'c', 'may residence tower', 'jade residence', 'riviera residence',
    'g', 'e', 'taya residences', 'vida residences', 'dana tower', 'd', 'building'
]

# Filter matches excluding unwanted prefixes
valid_matches = propagation_df[
    ~propagation_df['unmatched_proj_prefix'].str.lower().isin(excluded_prefixes)
].copy()

# Apply mapping to df_dev
mapping_dict = dict(zip(valid_matches['unmatched_proj_prefix'], valid_matches['developer_suggested']))

df_dev['developer_clean'] = df_dev['developer_clean']  # keep existing values
df_dev.loc[
    df_dev['proj_prefix'].isin(mapping_dict.keys()) & df_dev['developer_clean'].isna(),
    'developer_clean'
] = df_dev['proj_prefix'].map(mapping_dict)

final_cov = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage: {final_cov:.1f}%")

Final developer coverage: 68.9%


## 7.2. Buildings

In [48]:
# Apply to your DataFrame
df_dev[['building_prefix', 'building_suffix']] = df_dev['building'].apply(split_project_name)

In [49]:
# Get known (already matched) and unknown projects
known_buildings = (
    df_dev[df_dev['developer_clean'].notna()]
    [['building_prefix', 'developer_clean']]
    .drop_duplicates(subset='building_prefix')
)
unknown_buildings = (
    df_dev[df_dev['developer_clean'].isna()]
    .loc[df_dev['building_prefix'].notna() & (df_dev['building_prefix'].str.strip() != ""), ['building_prefix']]
    .drop_duplicates(subset='building_prefix')
)

# For each unknown project, find the most similar known project
matches = []
for build in unknown_buildings['building_prefix']:
    match = process.extractOne(
        build,
        known_buildings['building_prefix'],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=90  # adjust threshold; 90 = strong match
    )
    if match:
        matched_build, score, _ = match
        matched_dev = known_buildings.loc[known_buildings['building_prefix']==matched_build, 'developer_clean'].iloc[0]
        matches.append((build, matched_build, matched_dev, score))

propagation_df = pd.DataFrame(matches, columns=['unmatched_building_prefix','matched_building_prefix','developer_suggested','similarity'])
print(f"{len(propagation_df)} unmatched project-prefixes have strong similarity (>=95) with known project-prefixes.")
display(propagation_df.head())

28 unmatched project-prefixes have strong similarity (>=95) with known project-prefixes.


,unmatched_building_prefix,matched_building_prefix,developer_suggested,similarity
0,saba,saba,saba properties,100.000000
1,the residences ii,the residences,al habtoor,90.322581
2,balqis residence,bali residences,serene developments,90.322581
3,elle residences,elite residences,tameer holdings,90.322581
4,serra tower,terra tower,dugasta properties,90.909091


In [50]:
# your exclusion list
excluded_prefixes = [
    'balqis residence', 'elle residences', 'axis residences', 'the', 'amalia residences',
    'laya residences', 'olivia residences', 'verdana', 'liv residence', 'the residence',
    'maya residences', 'maya townhouses', 'serra tower', 'prive residence',
    'f', 'c', 'may residence tower', 'jade residence', 'riviera residence',
    'g', 'e', 'taya residences', 'vida residences', 'dana tower', 'd', 'building'
]

# Filter matches excluding unwanted prefixes
valid_matches = propagation_df[
    ~propagation_df['unmatched_building_prefix'].str.lower().isin(excluded_prefixes)
].copy()

# Apply mapping to df_dev
mapping_dict = dict(zip(valid_matches['unmatched_building_prefix'], valid_matches['developer_suggested']))

df_dev['developer_clean'] = df_dev['developer_clean']  # keep existing values
df_dev.loc[
    df_dev['building_prefix'].isin(mapping_dict.keys()) & df_dev['developer_clean'].isna(),
    'developer_clean'
] = df_dev['building_prefix'].map(mapping_dict)

final_cov = df_dev['developer_clean'].notna().mean() * 100
print(f"Final developer coverage: {final_cov:.1f}%")

Final developer coverage: 69.0%


# 8. Final Validation

# 9. Export

In [52]:
# Adapting the original df to merge
df['proj_concat'] = (
    df['master_project'] + ' | ' +
    df['project'] + ' | ' +
    df['building']
).str.strip()

In [53]:
# Keeping only project and developer_clean
df_dev_clean = df_dev[['proj_concat', 'developer_clean']].drop_duplicates(subset='proj_concat')

In [54]:
# Merging developer info into main dataset
df_merged = df.merge(df_dev_clean, on='proj_concat', how='left')

# Checking coverage after merge
coverage = df_merged['developer_clean'].notna().mean()
print(f"Developer field coverage after merge: {coverage:.1%}")

Developer field coverage after merge: 69.0%


In [55]:
df_merged.head()

,transaction_type,date,property_type,registration,area,building,project,master_project,landmark,metro,...,rooms,parking,size,price,metre_price,no_sellers,no_buyers,no_third_parties,proj_concat,developer_clean
0,Sales,2025-05-27,Villa,Existing Properties,Mirdif,,,,Dubai International Airport,Etisalat Metro Station,...,6 B/R,0,1520.45,5500000.0,3617.35,1,1,0,| |,NaN
1,Mortgages,2025-10-16,Villa,Existing Properties,Mirdif,,,,Dubai International Airport,Rashidiya Metro Station,...,5 B/R,0,696.77,2750000.0,3946.78,1,1,0,| |,NaN
2,Gifts,2025-07-03,Villa,Existing Properties,Mirdif,,,,Dubai International Airport,Rashidiya Metro Station,...,6 B/R,0,6967.73,42500001.0,6099.55,1,1,0,| |,NaN
3,Sales,2025-01-23,Villa,Existing Properties,Abu Hail,,,,Dubai International Airport,Abu Baker Al Siddique Metro Station,...,4 B/R,0,231.64,1350000.0,5828.01,1,1,0,| |,NaN
4,Sales,2025-05-14,Unit,Off-Plan Properties,Burj Khalifa,volta tower,volta tower,,Burj Khalifa,Buj Khalifa Dubai Mall Metro Station,...,4 B/R,1,220.14,6659000.0,30248.93,1,1,0,| volta tower | volta tower,NaN


In [56]:
# Removing building and master_project columns
df_merged = df_merged.drop(columns=['building', 'master_project','proj_concat'], errors='ignore')

In [57]:
df_merged.head()

,transaction_type,date,property_type,registration,area,project,landmark,metro,mall,rooms,parking,size,price,metre_price,no_sellers,no_buyers,no_third_parties,developer_clean
0,Sales,2025-05-27,Villa,Existing Properties,Mirdif,,Dubai International Airport,Etisalat Metro Station,City Centre Mirdif,6 B/R,0,1520.45,5500000.0,3617.35,1,1,0,NaN
1,Mortgages,2025-10-16,Villa,Existing Properties,Mirdif,,Dubai International Airport,Rashidiya Metro Station,City Centre Mirdif,5 B/R,0,696.77,2750000.0,3946.78,1,1,0,NaN
2,Gifts,2025-07-03,Villa,Existing Properties,Mirdif,,Dubai International Airport,Rashidiya Metro Station,City Centre Mirdif,6 B/R,0,6967.73,42500001.0,6099.55,1,1,0,NaN
3,Sales,2025-01-23,Villa,Existing Properties,Abu Hail,,Dubai International Airport,Abu Baker Al Siddique Metro Station,Missing,4 B/R,0,231.64,1350000.0,5828.01,1,1,0,NaN
4,Sales,2025-05-14,Unit,Off-Plan Properties,Burj Khalifa,volta tower,Burj Khalifa,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,4 B/R,1,220.14,6659000.0,30248.93,1,1,0,NaN


In [58]:
# Replacing missing project names with 'missing'
df_merged['project'] = df_merged['project'].replace('', pd.NA)  # make sure empty strings become NA first
df_merged['project'] = df_merged['project'].fillna('missing')
df_merged['project'].value_counts()

project
missing              22960
binghatti skyrise     2673
sobha solis           2066
binghatti elite       1690
skyvue                1620
                     ...  
fairway villas 3         1
j-haus residences        1
haven villas             1
arib collection          1
park villa's             1
Name: count, Length: 2379, dtype: int64

In [59]:
# Renaming developer column and replace missing values
df_merged = df_merged.rename(columns={'developer_clean': 'developer'})
df_merged['developer'] = df_merged['developer'].fillna('missing')
df_merged['developer'].value_counts()

developer
missing                   68380
binghatti                 19438
emaar                     17506
sobha                     11847
damac                     10892
                          ...  
bentley home                  6
nexus                         4
al marina                     4
newbury developments          1
casa vista development        1
Name: count, Length: 198, dtype: int64

In [60]:
# Saving the final dataset
#df_merged.to_csv("re_2025_analysis.csv", index=False)
#print("✅ Final dataset saved as 're_2025_analysis.csv'")